# $\pi/\sqrt3$: historical CMF bounds

Each section constructs a CMF, walks its trajectory, and prints $\delta_N$ at source depth 1000 for the reduced rational approximant. A finite-depth value does not prove the published measure bound.

In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 python-flint mpmath==1.3.0 matplotlib==3.11.2

## Setup

The CMFs are constructed directly by `ramanujantools.cmf.pfq.pFq`. The starting readouts, walks, and approximants are exact.

In [ ]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx

from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
with ctx.workdps(7000):
    TARGET = arb.pi()/arb(3).sqrt()
DEPTHS = list(range(10, 1001, 10))
x0, x1, y0 = sp.symbols("x0 x1 y0")
results = {}


def delta(approximant):
    """Finite-depth exponent of the reduced rational approximant."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2*q.bit_length() + 128):
        error = abs(TARGET - arb(p)/q)
        assert error > 0, "Increase TARGET and working precision"
        return float((-1 - error.log()/arb(q).log()).mid())

## 1. Alladi–Robinson family: $\pi/\sqrt3$

[Quadratic integral and measure formula](https://www.math.ru.nl/~wzudilin/gat/oALLADI4.txt).
$$E_n=\int_0^1\frac{t^n(1-t)^n}{(t^2+3)^{n+1}}\,dt.$$
**CMF:** rank 2, ${}_2F_1(-1/3)$ ? **Start:** $(1/2,1;3/2)$ ? **Direction:** $(1,1;2)$. One step advances the source index by two; the plotted samples use $N/2$ steps.
With $c=\arctan(1/\sqrt{3})/\sqrt{3}$, $E_n=C_n+B_nc$ and $p_n/q_n=-6C_n/B_n$. The source labels $c$ differently; the integral fixes its value. The displayed measure formula also uses its denominator lemma.
The exact identity
$$E_n=\frac{(n!)^2}{(2n+1)!\,3^{n+1}}{}_2F_1\!\left(\frac{n+1}{2},\frac{n+2}{2};n+\frac32;-\frac1{3}\right)$$
follows by expanding the quadratic denominator and cancelling an equal upper and lower parameter. The common scalar cancels in the rational readout.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 3))
start = {x0: sp.Rational(1, 2), x1: 1, y0: sp.Rational(3, 2)}
trajectory = {x0: 1, x1: 1, y0: 2}
# One lattice step advances the source index by two; all sampled N are even.
walks = cmf.walk(trajectory, [N//2 for N in DEPTHS], start)
approximants = [-6*M[1, 0]/(4*(2*M[0, 0]-M[1, 0])) for M in walks]

deltas = [delta(r) for r in approximants]
results["Alladi–Robinson (1980)"] = (DEPTHS, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## 2. Heimonen–Matala-aho–Väänänen (1994): $\beta=3/2$

[Jacobi Padé construction, §§3–4 and examples](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/249C98E58A8F5967C24F7999398BDBCE/S0004972700013691a.pdf/an-application-of-jacobi-type-polynomials-to-irrationality-measures.pdf). The published bound is $\mu_0=5.0874\ldots$.
Let $F(z)={}_2F_1(1,\tfrac12;\tfrac32;z)$ and $m=\lfloor(3/2)n\rfloor$. The Padé remainder is proportional to
$$J_0(n-\tfrac12,m,n+1)=\int_0^1 \frac{t^{n-1/2}(1-t)^m}{(1+t/3)^{n+1}}\,dt.$$
**CMF:** rank 2, ${}_2F_1(-1/3)$ ? **Start:** $(3,5/2;13/2)$, source $(n,m)=(2,3)$ ? **Direction:** $(2,2;5)$. The code gives the coefficient rows in the $(F,\theta F)$ basis; the target readout is $-2(CM)_0/(BM)_0$.
Euler integration gives $J_0=B(U+1,V+1){}_2F_1(W,U+1;U+V+2;-1/a)$. The beta factor cancels in the readout.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 3))
start = {x0: 3, x1: sp.Rational(5, 2), y0: sp.Rational(13, 2)}
C = Matrix([[-sp.Rational(28404, 5), sp.Rational(287469, 10)]])
B = Matrix([[6264, -31698]])
trajectory = {x0: 2, x1: 2, y0: 5}
periods = list(range(4, 500, 5))
depths = [2 + 2*k for k in periods]
walks = cmf.walk(trajectory, periods, start)
approximants = [-2*(C*M)[0]/(B*M)[0] for M in walks]
deltas = [delta(r) for r in approximants]
results["Heimonen–Matala-aho–Väänänen (1994)"] = (depths, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## 3. Hata (1993): three-root integral

[Theorem 1.2 and Section 4, pp. 337, 345-347](https://matwbn.icm.edu.pl/ksiazki/aa/aa63/aa6344.pdf)
give $\mu(\pi/\sqrt3)\le4.601579125506\ldots$.
With $\lambda=(1+i\sqrt3)/2$ and $h=(1+\lambda)/2$, Hata uses
$$I_n=\int_1^\lambda\frac{[(z-1)(z-\lambda)(z-h)]^{2n}}{z^{3n+1}}\,dz.$$
The substitution $z=h(1+t)$ gives $I_n=h^{3n}J_0(2n,2n,3n+1)$, where
$$J_j(u,v,w)=\int_{-i/\sqrt3}^{i/\sqrt3}
\frac{t^{v+j}(t^2+1/3)^u}{(1+t)^w}\,dt.$$
The contour avoids $t=-1$ and uses the branch with logarithmic endpoint difference $i\pi/3$.

**CMF:** rank 3, ${}_3F_2(-1/3)$; **Start:** $(1/2,1,1/2;1/2,3/2)$; **Direction:** $(3,3,2;0,6)$.
After setting $t=ix/\sqrt3$ and taking the even part on $[-1,1]$, the source integral is a common nonzero scalar times
$${}_3F_2\!\left(\frac{3n+1}{2},\frac{3n+2}{2},n+\frac12;\frac12,3n+\frac32;-\frac13\right).$$
Thus $N/2$ steps give the sampled even source indices $n=N$, including exactly $n=1000$.
In the $(F,\theta F,\theta^2F)$ basis, use $C=(0,12,-9)$ and $B=(16,-8,4)$.
The readout is $p_N/q_N=-(CM)_0/(BM)_0$.

`local_checks/verify_hata_pi_sqrt3.py` checks all seed moments by exact integration and both
coefficient rows against Hata's Laurent expansion (3.1) for $n=1,\ldots,6$.
The first approximants are $185/102$ and $149711/82540$.
The saved interval check at $n=1000$ uses 22134 and 33265 bits.
The published bound also uses Hata's denominator lemma and asymptotic estimates;
this recovery verifies the construction, not those proofs independently.


In [ ]:
x2, y1 = sp.symbols("x2 y1")
cmf = pFq(3, 2, -sp.Rational(1, 3))
start = {x0: sp.Rational(1, 2), x1: 1, x2: sp.Rational(1, 2),
         y0: sp.Rational(1, 2), y1: sp.Rational(3, 2)}
trajectory = {x0: 3, x1: 3, x2: 2, y0: 0, y1: 6}
walks = cmf.walk(trajectory, [N//2 for N in DEPTHS], start)
C = Matrix([[0, 12, -9]])
B = Matrix([[16, -8, 4]])
approximants = [-(C*M)[0]/(B*M)[0] for M in walks]

deltas = [delta(r) for r in approximants]
results["Hata (1993)"] = (DEPTHS, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")


delta at depth 1000: 0.276866661134


## All constructions

$\delta_N$ against the source depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published upper bound $\mu_0$ | $\delta_{\rm eq}$ | $\delta_{1000}$ | CMF |
|:--|--:|--:|--:|:--|
| Alladi–Robinson | 8.309986340 | 0.136799161 | 0.145170648232 | rank-2 ${}_2F_1$ walk |
| Heimonen–Matala-aho–Väänänen | $5.0874\ldots$ | $\approx0.2447$ | 0.255217023998 | rank-2 ${}_2F_1$ walk |
| Hata (1993) | $4.601579125506\ldots$ | 0.277655985098 | 0.276866661134 | rank-3 ${}_3F_2$ walk |

$\delta_{\rm eq}=1/(\mu_0-1)$; the finite-depth values are diagnostics.